# MediConfirm
Check medication packages, have an optional conversation, and save explicit intake self-reports. The Notebook and desktop website share one medication list, settings file and history on this computer.


## 1. Install dependencies
Run this cell once in your Anaconda kernel. Restart the kernel after installation, then continue with section 2. Whisper may download its model on first use.

In [ ]:
import sys
import subprocess
from pathlib import Path

PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src" / "mediconfirm").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the mediconfirm_v2 folder.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], cwd=PROJECT_DIR)
print("Installation complete. Restart the kernel and continue with section 2.")

## 2. Load API configuration
The existing key and model are stored in `config.local.json`, which is excluded from Git. No key entry is needed for each run. On a different computer, copy `config.example.json` to `config.local.json` and fill in the key once. The `HKBU_API_KEY` environment variable can override the local key.

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src" / "mediconfirm").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the mediconfirm_v2 folder.")
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
from mediconfirm.shared_storage import prepare_shared_storage
prepare_shared_storage(PROJECT_DIR)
config_path = PROJECT_DIR / "config.local.json"
config = json.loads(config_path.read_text()) if config_path.exists() else {}
API_KEY = os.environ.get("HKBU_API_KEY", config.get("api_key", "")).strip()
MODEL_NAME = config.get("model", "gpt-4.1")
API_VERSION = config.get("api_version", "2025-04-01-preview")
if not API_KEY or API_KEY.startswith("YOUR_"):
    raise ValueError("Set your key once in config.local.json or HKBU_API_KEY.")
print("API configuration loaded. Model:", MODEL_NAME)

## 3. Medication list
Enter the medication name, strength and dose. Choose hours and minutes using the scrolling selectors, and use **+ Add intake time** for additional daily times. Save the list before checking packages.


In [ ]:
import importlib
from IPython.display import display
from mediconfirm import bulk_medications, medication_form

importlib.reload(bulk_medications)
importlib.reload(medication_form)
if "medication_form_widget" in globals():
    medication_form_widget.close()
medication_form_widget = medication_form.MedicationListForm(PROJECT_DIR / "data" / "personal_medications.json")
display(medication_form_widget.widget)

## 4. Run the assistant
Personal settings provides two independent delays: ask again after **300 seconds** (5 minutes), and mark Unconfirmed and create a family notification after **600 seconds** (10 minutes). Both can be changed; the repeat reminder must come first. Changes apply to new events.

The Notebook and website use the same saved settings and JSON log. Refresh the medication form after editing the list in another interface. No separate test logs are created from the user interface. Real email alerts use the team Gmail sender after one-time local configuration and explicit enablement in the website. Without configuration, notifications are recorded as not sent.

Start a medication check, capture one package, and explicitly report intake only after a match.


In [ ]:
from IPython.display import display
import importlib
from mediconfirm import integrations, simple_service, drug_sources, dialogue, speech_tools, intake_store, plan_actions, simple_ui, web_settings, notebook_dashboard

if "_schedule_subscription" in globals():
    _schedule_subscription()
if globals().get("notebook_controls") is not None:
    notebook_controls.close()
elif globals().get("simple_app") is not None:
    simple_app.close()
if globals().get("time_selector") is not None:
    time_selector.close()
if "_schedule_panel" in globals():
    _schedule_panel.close()
for module in (integrations, simple_service, drug_sources, dialogue, speech_tools, intake_store, plan_actions, simple_ui, web_settings, notebook_dashboard):
    importlib.reload(module)
if "medication_form_widget" not in globals():
    raise ValueError("Run section 3 first.")

medications = medication_form_widget.medications
simple_app = simple_ui.SimpleCameraApp(
    api_key=API_KEY, model=MODEL_NAME, api_version=API_VERSION,
    medications=medications, selected_time=medications[0]["time"] if medications else "", voice=True,
    log_path=PROJECT_DIR / "data" / "intake_records.json",
)
notebook_controls = notebook_dashboard.NotebookDashboard(simple_app, PROJECT_DIR / "data")
_schedule_subscription = medication_form_widget.on_saved(notebook_controls.set_medications)
display(notebook_controls.widget)


## Scope

Records are self-reports. Unconfirmed does not establish a missed dose. Severe-symptom handling covers predefined English phrases, not medical triage. General-use explanations require a verified DailyMed source. Notebook reminders require a running kernel; long-running cells can delay checks. Web reminders require the Python server and an awake computer. Saved deadlines are reconciled on reopening. The prototype does not diagnose, change prescriptions, or run as a phone app.

## 5. Desktop website
Run `python main.py web` and open the Computer link. Home shows today’s medication plan and starts a package check. Chat provides optional conversation. Me contains the medication list, reminder settings, history and family notifications.

All browsers and this Notebook share `data/personal_medications.json`, `data/settings.json` and `data/intake_records.json`. Restart the Notebook kernel after updating the source modules.
